# ⚡ Single XGBoost Model

This notebook is the result of rigorous A/B testing of various feature engineering techniques discussed in the community. By filtering out synthetic generator noise, leveraging interaction pairs, and strictly isolating Target Encoding inside the cross-validation loop to prevent data leakage, this single XGBoost model achieves a highly competitive Leaderboard score.

### 🙏 Acknowledgements & Credits
Special thanks to the community members whose research, notebooks, and forum posts directly shaped this approach:
*   **Chris Deotte** for validating the [Fable 5.1 - EDA - Original Data Insights](https://www.kaggle.com/code/cdeotte/fable-5-1-eda-original-data-insights) and [Fable 5.1 - XGB Starter](https://www.kaggle.com/code/cdeotte/fable-5-1-xgb-starter).
*   **Dmitriy Kolmikov** for validating the [Target Encoding of Flight Distances](https://www.kaggle.com/competitions/playground-series-s6e10/discussion/747742).
*   **Sarun P M** for the brilliant [Route-Based Profiling](https://www.kaggle.com/code/sarunpm/airline-satisfaction-lgbm-0-96025-lb) logic.
*   [**Shelton Wang**](https://www.kaggle.com/code/wangxintong111/s6e10-lightgbm-context-stats-cv-0-96065) & [**Evgen Dvorkin**](https://www.kaggle.com/code/evgendvorkin/s6e10-lightgbm-cv-5-folds-0-96111) for the Persona-Context Interaction concept.
*   **Starkhushi** for the [Statistical Field Guide](https://www.kaggle.com/competitions/playground-series-s6e10/discussion/745932), which helped us identify and remove noisy columns.

**If you find this notebook helpful, please consider upvoting the linked resources above as well!**

In [1]:
import os, gc
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import TargetEncoder
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings('ignore')
os.environ['XGBOOST_WARN_PREDICT_FALLBACK'] = '0'

# 1. LOAD DATA

In [2]:
DATA_DIR = '/kaggle/input/competitions/playground-series-s6e10'
train = pd.read_csv(f'{DATA_DIR}/train.csv')
test = pd.read_csv(f'{DATA_DIR}/test.csv')
sub = pd.read_csv(f'{DATA_DIR}/sample_submission.csv')

TARGET = 'satisfaction'
ID = 'id'
EXPERIMENT_NAME = "XGBoost"
train[TARGET] = train[TARGET].astype(int)
NJ_SEED = 42

for df in [train, test]:
    df['Arrival Delay in Minutes'] = df['Arrival Delay in Minutes'].fillna(df['Departure Delay in Minutes'])

CAT_FEATURES = ['Gender', 'Customer Type', 'Type of Travel', 'Class']
RATING_COLS = [
    'Inflight wifi service', 'Departure/Arrival time convenient', 'Ease of Online booking', 
    'Food and drink', 'Online boarding', 'Seat comfort', 'Inflight entertainment', 
    'On-board service', 'Leg room service', 'Baggage handling', 'Checkin service', 'Cleanliness'
]

# 2.  FEATURE ENGINEERING

In [3]:
# 🧮 Generating Domain Features & Pairs
for df in [train, test]:
    df['biz_elite_nj'] = ((df['Type of Travel'] == 'Business travel') & (df['Class'] == 'Business')).astype(np.int8)
    df['Wifi_Booking_pair'] = df['Inflight wifi service'].astype(str) + "_" + df['Ease of Online booking'].astype(str)
    df['Wifi_Time_pair']    = df['Inflight wifi service'].astype(str) + "_" + df['Departure/Arrival time convenient'].astype(str)
    df['Boarding_Wifi_pair']= df['Online boarding'].astype(str) + "_" + df['Inflight wifi service'].astype(str)
    df['Class_Travel_pair'] = df['Class'].astype(str) + "_" + df['Type of Travel'].astype(str)
    df['Cust_Travel_pair']  = df['Customer Type'].astype(str) + "_" + df['Type of Travel'].astype(str)
    df['Class_Cust_pair']   = df['Class'].astype(str) + "_" + df['Customer Type'].astype(str)
    df['Class_Wifi_pair']     = df['Class'].astype(str) + "_" + df['Inflight wifi service'].astype(str)
    df['Class_Boarding_pair'] = df['Class'].astype(str) + "_" + df['Online boarding'].astype(str)
    df['Travel_Wifi_pair']    = df['Type of Travel'].astype(str) + "_" + df['Inflight wifi service'].astype(str)
    df['Travel_Boarding_pair']= df['Type of Travel'].astype(str) + "_" + df['Online boarding'].astype(str)
    df['Seat_Ent_pair']       = df['Seat comfort'].astype(str) + "_" + df['Inflight entertainment'].astype(str)
    df['Route_Class_pair'] = df['Flight Distance'].astype(str) + "_" + df['Class'].astype(str)
    df['Route_Travel_pair'] = df['Flight Distance'].astype(str) + "_" + df['Type of Travel'].astype(str)

INTERACTION_COLS = [
    'Wifi_Booking_pair', 'Wifi_Time_pair', 'Boarding_Wifi_pair', 'Class_Travel_pair', 'Cust_Travel_pair', 
    'Class_Cust_pair', 'Class_Wifi_pair', 'Class_Boarding_pair', 'Travel_Wifi_pair', 'Travel_Boarding_pair', 
    'Seat_Ent_pair', 'Route_Class_pair', 'Route_Travel_pair'
]

# 🔢 Extracting  Digits
DIGIT_MAP = {'Age': [0, 1], 'Flight Distance': [0, 1, 2, 3], 'Departure Delay in Minutes': [1], 'Arrival Delay in Minutes': [1]}
digit_feats_nj = [] 
for c, powers in DIGIT_MAP.items():
    for k in powers:
        col_name = f"{c}_digit{k}"
        train[col_name] = (train[c].fillna(0) // (10**k) % 10).astype(np.int8)
        test[col_name] = (test[c].fillna(0) // (10**k) % 10).astype(np.int8)
        digit_feats_nj.append(col_name)

# ✈️ Extracting Route Profiles based on Flight Distance
for df in [train, test]:
    df['Route_ID'] = df['Flight Distance'].astype(int)

route_counts = pd.concat([train['Route_ID'], test['Route_ID']], ignore_index=True).value_counts()
for df in [train, test]:
    df['Route_Freq'] = df['Route_ID'].map(route_counts).astype(np.int32)

route_profile_cols = ['Age', 'Departure Delay in Minutes'] + RATING_COLS
route_profile = train.groupby('Route_ID')[route_profile_cols].mean()
route_profile.columns = [f"Route_Mean_{c.replace(' ', '_').replace('/', '_')}" for c in route_profile.columns]
route_profile = route_profile.reset_index()

train = train.merge(route_profile, on="Route_ID", how="left")
test  = test.merge(route_profile,  on="Route_ID", how="left")

for c in route_profile.columns:
    if c == "Route_ID": continue
    train[c] = train[c].fillna(train[c].mean())
    test[c]  = test[c].fillna(train[c].mean())

train.drop(columns=['Route_ID'], inplace=True)
test.drop(columns=['Route_ID'], inplace=True)

# Setup Encoding
TARGET_ENCODE_COLS = CAT_FEATURES + RATING_COLS + digit_feats_nj + INTERACTION_COLS

num_to_cat_cols = []
for c in ['Age', 'Flight Distance']: 
    cat_name = f"{c}_cat"
    train[cat_name] = train[c].fillna(-1).astype(str)
    test[cat_name] = test[c].fillna(-1).astype(str)
    num_to_cat_cols.append(cat_name)

TARGET_ENCODE_COLS.extend(num_to_cat_cols)

for df in [train, test]:
    for col in CAT_FEATURES + digit_feats_nj:
        df[col] = df[col].astype(str)
    for col in RATING_COLS:
        df[col] = "Rating_" + df[col].astype(str)

# 📊 Applying Frequency Encoding
combined_nj = pd.concat([train, test], ignore_index=True)
for col in TARGET_ENCODE_COLS:
    freq_mapping = combined_nj[col].value_counts(normalize=True).to_dict()
    train[f"{col}_FE"] = train[col].map(freq_mapping).astype(np.float32)
    test[f"{col}_FE"]  = test[col].map(freq_mapping).astype(np.float32)
del combined_nj; gc.collect()

# Hardcoded Pruning List
FEATURES_TO_DROP = [
    'Departure Delay in Minutes_digit1_FE', 'Departure Delay in Minutes_digit1_TE_auto',
    'Arrival Delay in Minutes_digit1_FE', 'Arrival Delay in Minutes_digit1_TE_auto',
    'Departure Delay in Minutes', 'Flight Distance_digit3_FE', 'Flight Distance_digit1_FE',
    'Age_digit0_FE', 'Flight Distance_digit0_FE', 'Flight Distance_digit2_FE'
]

# 3. TRAINING StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [4]:
FEATURES = [c for c in train.columns if c not in [ID, TARGET, 'Arrival_Delay_missing'] + TARGET_ENCODE_COLS]
X = train[FEATURES + TARGET_ENCODE_COLS]
y = train[TARGET]
X_test = test[FEATURES + TARGET_ENCODE_COLS]

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=NJ_SEED)
oof_preds = np.zeros(len(train))
test_preds = np.zeros(len(test))
fold_scores = []
best_iters = []

xgb_params = {
        'n_estimators': 10000,
        'learning_rate': 0.005,
        'max_depth': 9,
        'subsample': 0.85,
        'colsample_bytree': 0.4,
        'colsample_bynode': 0.65,
        'min_child_weight': 30,
        'reg_lambda': 2.5,
        'max_bin': 256,
        'tree_method': 'hist',
        'device': 'cuda',       
        'eval_metric': 'auc',
        'early_stopping_rounds': 200,
        'random_state': NJ_SEED
    }

for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y), 1):
    X_train, y_train = X.iloc[train_idx].copy(), y.iloc[train_idx]
    X_valid, y_valid = X.iloc[valid_idx].copy(), y.iloc[valid_idx]
    X_test_fold = X_test.copy()

    te_auto = TargetEncoder(shuffle=True, cv=5, smooth='auto', random_state=NJ_SEED)
    X_train_enc = te_auto.fit_transform(X_train[TARGET_ENCODE_COLS], y_train)
    X_valid_enc = te_auto.transform(X_valid[TARGET_ENCODE_COLS])
    X_test_enc  = te_auto.transform(X_test_fold[TARGET_ENCODE_COLS])

    for i, col in enumerate(TARGET_ENCODE_COLS):
        X_train[f"{col}_TE_auto"] = X_train_enc[:, i].astype(np.float32)
        X_valid[f"{col}_TE_auto"] = X_valid_enc[:, i].astype(np.float32)
        X_test_fold[f"{col}_TE_auto"] = X_test_enc[:, i].astype(np.float32)
        X_train.drop(columns=[col], inplace=True)
        X_valid.drop(columns=[col], inplace=True)
        X_test_fold.drop(columns=[col], inplace=True)

    del X_train_enc, X_valid_enc, X_test_enc; gc.collect()
    
    X_train.drop(columns=FEATURES_TO_DROP, inplace=True, errors='ignore')
    X_valid.drop(columns=FEATURES_TO_DROP, inplace=True, errors='ignore')
    X_test_fold.drop(columns=FEATURES_TO_DROP, inplace=True, errors='ignore')


    model = xgb.XGBClassifier(**xgb_params)
    
    model.fit(
        X_train, y_train, 
        eval_set=[(X_valid, y_valid)], 
        verbose=1000,
    )

    
    valid_probs = model.predict_proba(X_valid)[:, 1]
    oof_preds[valid_idx] = valid_probs
    test_preds += model.predict_proba(X_test_fold)[:, 1] / skf.n_splits
    
    fold_auc = roc_auc_score(y_valid, valid_probs)
    fold_scores.append(fold_auc)
    best_iter = model.best_iteration if hasattr(model, 'best_iteration') else 0
    best_iters.append(best_iter)
    print(f"   --> Fold {fold} | Best Iter: {best_iter} | ROC-AUC: {fold_auc:.5f}")

mean_auc = np.mean(fold_scores)
print("\n" + "-"*40)
print(f"🏆 {EXPERIMENT_NAME} FINAL CV: {mean_auc:.5f} ± {np.std(fold_scores):.5f}")
print("-" * 40)

[0]	validation_0-auc:0.95294
[1000]	validation_0-auc:0.96023
[2000]	validation_0-auc:0.96105
[3000]	validation_0-auc:0.96124
[4000]	validation_0-auc:0.96130
[5000]	validation_0-auc:0.96133
[5202]	validation_0-auc:0.96132
   --> Fold 1 | Best Iter: 5002 | ROC-AUC: 0.96133
[0]	validation_0-auc:0.95215
[1000]	validation_0-auc:0.95887
[2000]	validation_0-auc:0.95968
[3000]	validation_0-auc:0.95987
[3986]	validation_0-auc:0.95992
   --> Fold 2 | Best Iter: 3786 | ROC-AUC: 0.95992
[0]	validation_0-auc:0.95392
[1000]	validation_0-auc:0.96065
[2000]	validation_0-auc:0.96147
[3000]	validation_0-auc:0.96164
[3577]	validation_0-auc:0.96167
   --> Fold 3 | Best Iter: 3377 | ROC-AUC: 0.96167
[0]	validation_0-auc:0.95378
[1000]	validation_0-auc:0.96021
[2000]	validation_0-auc:0.96095
[3000]	validation_0-auc:0.96107
[3590]	validation_0-auc:0.96109
   --> Fold 4 | Best Iter: 3390 | ROC-AUC: 0.96109
[0]	validation_0-auc:0.95347
[1000]	validation_0-auc:0.96010
[2000]	validation_0-auc:0.96067
[3000]	vali

# 4. SAVE SUBMISSION AND OOF PREDICTIONS

In [5]:
pd.DataFrame({ID: train[ID], 'oof_pred': oof_preds}).to_csv(f"OOF_{EXPERIMENT_NAME}.csv", index=False)
pd.DataFrame({ID: test[ID], TARGET: test_preds}).to_csv(f"TEST_CV_{EXPERIMENT_NAME}.csv", index=False)
sub[TARGET] = test_preds
sub.to_csv(f"SUB_CV_{EXPERIMENT_NAME}.csv", index=False)
print("💾 CV Results and Artifacts saved.")

💾 CV Results and Artifacts saved.
